# Simulation of Optical Coherent System for 16-QAM Single Channel Dual Polarization


In [1]:
import numpy as np
from optic.models.devices import mzm, photodiode, edfa, iqm, coherentReceiver, pdmCoherentReceiver, basicLaserModel
from optic.models.channels import linearFiberChannel, ssfm
from optic.comm.modulation import modulateGray, grayMapping
from optic.comm.sources import bitSource, symbolSource
from optic.dsp.core import upsample, pulseShape, pnorm, anorm, signalPower, firFilter, decimate, symbolSync,phaseNoise

try:
    from optic.dsp.coreGPU import checkGPU
    if checkGPU():
        from optic.dsp.coreGPU import firFilter
    else:
        from optic.dsp.core import firFilter
except ImportError:
    from optic.dsp.core import firFilter

from optic.utils import parameters, dBm2W, ber2Qfactor
from optic.plot import eyediagram, pconst, plotPSD
import matplotlib.pyplot as plt
from scipy.special import erfc
from tqdm.notebook import tqdm
import scipy as sp
import scipy.constants as const

try:
    from optic.models.modelsGPU import manakovSSF
except:
    from optic.models.channels import manakovSSF

from optic.dsp.equalization import edc, mimoAdaptEqualizer, ffe
from optic.dsp.carrierRecovery import cpr
from optic.comm.metrics import fastBERcalc, monteCarloGMI, monteCarloMI, calcEVM, bert
from optic.dsp.clockRecovery import gardnerClockRecovery


import logging as logg
logg.basicConfig(level=logg.INFO, format='%(message)s', force=True)
import time

In [2]:
from IPython.core.display import HTML
from IPython.core.pylabtools import figsize

HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

In [3]:
"""
Single-channel 16-QAM coherent transmitter using OptiCommPy's IQM
(patterned after optic.models.tx.simpleWDMTx).

What you get:
- symbTx: 16-QAM symbols (unit-energy constellation by design)
- sigTx: pulse-shaped complex baseband (normalized to peak amplitude 1)
- sigLO: complex optical carrier field (unit amplitude, optional phase noise)
- sigTxo: modulated optical field at the transmitter output, with desired launch power
"""



"\nSingle-channel 16-QAM coherent transmitter using OptiCommPy's IQM\n(patterned after optic.models.tx.simpleWDMTx).\n\nWhat you get:\n- symbTx: 16-QAM symbols (unit-energy constellation by design)\n- sigTx: pulse-shaped complex baseband (normalized to peak amplitude 1)\n- sigLO: complex optical carrier field (unit amplitude, optional phase noise)\n- sigTxo: modulated optical field at the transmitter output, with desired launch power\n"

In [ ]:
# -----------------------------------------------
# Transmitter Parameters & Array Allocation
#-------------------------------------------------

# 1) General Parameters
SpS = 16                 # samples per symbol
Rs = 32e9                # symbol rate [baud]
Fs = Rs * SpS            # sampling frequency [Hz]
M = 16                   # QPSK -> M=4, constType='qam'
nBits = 400000           # total bits to generate
rollOff = 0.01           # RRC roll-off
nFilterTaps = 1024       # RRC filter taps
mzmScale = 0.5           # IQM drive scale = "modulation depth" (0.5 is default in OptiCommPy)
Vpi =2                   # IQM pi Voltage  (2V is defualt in OpticommPy)
P_launch_dBm = 0         # desired launched optical power [dBm] -- for 4QAM try 6 dbm -- for 16QAM 0 was good
laserLinewidth = 100e3   # Hz (set e.g. 100e3 for phase noise), setting laserLineWidth to 0, models the ideal case 
nPolModes = 2            # Dual-Polarization
seed = 123               # seed for transmitter parts

# 2) Symbol Source Parameters
paramSymb = parameters()
paramSymb.nSymbols = int(nBits // np.log2(M))  # symbols = bits / log2(M)
paramSymb.M = M
paramSymb.constType = "qam"                    # 'qam' with M=4 -> QPSK
paramSymb.dist = "uniform"                     # uniform symbol probabilities
paramSymb.seed = seed
paramSymb.shapingFactor = 0

constSymb = grayMapping(paramSymb.M, paramSymb.constType)
if paramSymb.dist == "uniform":
   px = np.ones(paramSymb.M) / paramSymb.M
elif paramSymb.probDist == "maxwell-boltzmann":
   px = np.exp(-paramSymb.shapingFactor * np.abs(constSymb) ** 2)
   px = px / np.sum(px)
else:
   raise ValueError("Invalid probability distribution.")
paramSymb.px = px

# 3) UpSampling and FIR Parameters
paramPulse = parameters()
paramPulse.pulseType = "rrc"
paramPulse.nFilterTaps = nFilterTaps
paramPulse.rollOff = rollOff
paramPulse.SpS = SpS

# 4) IQM Parameters
paramIQM = parameters()
paramIQM.Vpi = Vpi
paramIQM.VbI = -Vpi
paramIQM.VbQ = -Vpi
paramIQM.Vphi = Vpi/2

# 5) Allocate Array
t = np.arange(0, int(paramSymb.nSymbols * SpS)) 
sigTx = np.zeros((len(t), nPolModes), dtype="complex")                             # stores the transmitted signal after pulse shaping (DAC)
symbTx = np.zeros((len(t) // SpS, nPolModes), dtype="complex")                     # stores the transmitted symbols
sigTxo = np.zeros(((nBits * SpS) // int(np.log2(M)), nPolModes), dtype="complex")  # stores the optically modulated signal, entering the channel

In [ ]:
for mode in range(nPolModes): # iteration 1: mode=0, iteration 2: mode=1

    if paramSymb.seed is not None:
        # iteration 1: seed = 123 and mode = 0
        # iteration 2: seed = 123 and mode = 1 -> therefore becomes seed = 124
        # the seed must be change between the two iterations -> the symbol sequences
        # of two polarizations are different 
        seed = paramSymb.seed + mode 
    else:
        seed = None

    # Generate the constellation symbols
    paramSymb.seed = seed
    symbTx_ = symbolSource(paramSymb)
    symbTx[:, mode] = symbTx_

    # Upsampling
    symbolsUp = upsample(symbTx_, SpS)

    # Create pulse shaping filter
    pulse = pulseShape(paramPulse)

    # Pulse shaping
    sigTx_ = firFilter(pulse, symbolsUp)
    sigTx_ = sigTx_ / np.max(np.abs(sigTx_))
    sigTx[:, mode] = sigTx_

    # Optical modulation
    if mode == 0:  # generate LO field with phase noise (ONLY ONCE, iteration 1 only)
        ϕ_pn_lo = phaseNoise(laserLinewidth, len(sigTx), 1 / Fs, seed=seed)
        sigLO = np.exp(1j * ϕ_pn_lo)

    u_drive = mzmScale * sigTx_
    sigTxo_ = iqm(sigLO, u_drive, paramIQM)

    # Set Lauch Power
    P_launch_W = dBm2W(P_launch_dBm)
    sigTxo_ = np.sqrt(P_launch_W) * pnorm(sigTxo_)
    sigTxo[:, mode] = sigTxo_